# Monitoramento da Pipeline de Dados

Este notebook consolida métricas operacionais e de qualidade da pipeline Literacy Data Pipeline.

O objetivo é fornecer uma visão centralizada sobre a execução das diferentes etapas do processamento, permitindo acompanhar volumes processados, status das execuções e indicadores de qualidade dos dados.

O monitoramento contempla as camadas Batch e Streaming e demonstra como a solução poderia evoluir para mecanismos de observabilidade em um ambiente produtivo.

## 1. Métricas de monitoramento

O monitoramento da solução considera quatro dimensões principais:

- **Volume:** quantidade de registros processados por dataset e camada;
- **Execução:** status e horário das execuções;
- **Qualidade:** quantidade de regras com status PASS, WARN e FAIL;
- **Streaming:** quantidade de eventos processados nas camadas Bronze e Silver.

Essas métricas permitem identificar alterações inesperadas de volume, falhas de processamento e problemas de qualidade antes do consumo analítico dos dados.

In [0]:
# Importa funções utilizadas na consolidação das métricas de monitoramento.

from pyspark.sql import functions as F
from datetime import datetime, timezone

In [0]:
# Define os caminhos das camadas monitoradas no Amazon S3.

BUCKET_NAME = "literacy-data-pipeline-tcf2-480749290106-us-east-1-an"

BRONZE_PATH = f"s3://{BUCKET_NAME}/bronze/"
SILVER_PATH = f"s3://{BUCKET_NAME}/silver/"
GOLD_PATH = f"s3://{BUCKET_NAME}/gold/"

## 2. Monitoramento de volume das camadas Batch

A primeira dimensão monitorada é o volume de dados processados.

A quantidade de registros é utilizada para identificar alterações inesperadas entre execuções, ausência de dados ou possíveis problemas durante o processamento.

In [0]:
# Define os datasets Batch da Silver monitorados pela pipeline.

DATASETS_SILVER = [
    "avaliacao_alfabetizacao_municipio",
    "avaliacao_alfabetizacao_uf",
    "avaliacao_alunos",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf"
]

# Define as tabelas analíticas finais da Gold.

TABELAS_GOLD = [
    "gold_brasil_desempenho",
    "gold_uf_desempenho",
    "gold_municipio_desempenho",
    "gold_municipio_evolucao",
    "gold_indicadores_alunos"
]

In [0]:
# Coleta as quantidades de registros das camadas Silver e Gold para compor a visão operacional da pipeline.

metricas_volume = []

for nome in DATASETS_SILVER:
    df = spark.read.parquet(f"{SILVER_PATH}{nome}/")

    metricas_volume.append({
        "camada": "silver",
        "objeto": nome,
        "registros": df.count(),
        "status": "SUCESSO"
    })

for nome in TABELAS_GOLD:
    df = spark.read.parquet(f"{GOLD_PATH}{nome}/")

    metricas_volume.append({
        "camada": "gold",
        "objeto": nome,
        "registros": df.count(),
        "status": "SUCESSO"
    })

df_monitoramento_volume = spark.createDataFrame(metricas_volume)

display(
    df_monitoramento_volume
    .orderBy("camada", "objeto")
)

## 3. Monitoramento do processamento Streaming

O fluxo Streaming é monitorado pela quantidade de eventos disponíveis nas camadas Bronze e Silver.

A comparação entre as duas camadas permite identificar possíveis perdas durante o processamento incremental e verificar se os eventos recebidos foram tratados e persistidos corretamente.

In [0]:
# Define os caminhos das saídas do pipeline de Streaming.

BRONZE_STREAM_PATH = (
    f"s3://{BUCKET_NAME}/bronze/streaming/avaliacao_alunos/"
)

SILVER_STREAM_PATH = (
    f"s3://{BUCKET_NAME}/silver/streaming/avaliacao_alunos/"
)

In [0]:
# Coleta as métricas de volume das camadas Bronze e Silver Streaming.

df_bronze_stream = spark.read.parquet(BRONZE_STREAM_PATH)
df_silver_stream = spark.read.parquet(SILVER_STREAM_PATH)

registros_bronze_stream = df_bronze_stream.count()
registros_silver_stream = df_silver_stream.count()

metricas_streaming = [
    {
        "camada": "bronze_streaming",
        "objeto": "avaliacao_alunos",
        "registros": registros_bronze_stream,
        "status": "SUCESSO"
    },
    {
        "camada": "silver_streaming",
        "objeto": "avaliacao_alunos",
        "registros": registros_silver_stream,
        "status": (
            "SUCESSO"
            if registros_bronze_stream == registros_silver_stream
            else "ATENCAO"
        )
    }
]

df_monitoramento_streaming = spark.createDataFrame(
    metricas_streaming
)

display(df_monitoramento_streaming)

In [0]:
# Verifica se todos os eventos da Bronze Streaming chegaram à Silver.

if registros_bronze_stream == registros_silver_stream:
    print(
        f"[OK] Streaming sincronizado: "
        f"{registros_bronze_stream} eventos processados."
    )
else:
    print(
        f"[ATENÇÃO] Divergência no Streaming: "
        f"Bronze={registros_bronze_stream} | "
        f"Silver={registros_silver_stream}"
    )

## 4. Monitoramento de Data Quality

O monitoramento de qualidade consolida indicadores críticos sobre os dados disponibilizados na camada Silver.

Nesta visão operacional são acompanhadas métricas que podem indicar problemas que exigem investigação, como duplicidade de chaves e inconsistências nos dados de alunos.

In [0]:
# Define as chaves utilizadas no monitoramento de duplicidades.

CHAVES_MONITORAMENTO = {
    "avaliacao_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "avaliacao_alfabetizacao_uf": ["ano", "sigla_uf", "rede"],
    "avaliacao_alunos": ["ano", "id_aluno"],
    "meta_alfabetizacao_brasil": ["ano", "rede"],
    "meta_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "meta_alfabetizacao_uf": ["ano", "sigla_uf", "rede"]
}

In [0]:
# Monitora duplicidades nas chaves principais dos datasets Silver.

metricas_qualidade = []

for nome, chaves in CHAVES_MONITORAMENTO.items():

    df = spark.read.parquet(
        f"{SILVER_PATH}{nome}/"
    )

    qtd_duplicadas = (
        df
        .groupBy(*chaves)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    metricas_qualidade.append({
        "dataset": nome,
        "metrica": "chaves_duplicadas",
        "valor": qtd_duplicadas,
        "status": "PASS" if qtd_duplicadas == 0 else "FAIL"
    })

In [0]:
# Monitora inconsistências críticas na base de alunos.

df_alunos = spark.read.parquet(
    f"{SILVER_PATH}avaliacao_alunos/"
)

binarios_invalidos = (
    df_alunos
    .filter(
        (~F.col("presenca").isin(0, 1))
        | (~F.col("preenchimento_caderno").isin(0, 1))
        | (~F.col("alfabetizado").isin(0, 1))
    )
    .count()
)

inconsistencias_caderno = (
    df_alunos
    .filter(
        (F.col("preenchimento_caderno") == 0)
        & (
            F.col("proficiencia").isNotNull()
            | F.col("peso_aluno").isNotNull()
        )
    )
    .count()
)

metricas_qualidade.extend([
    {
        "dataset": "avaliacao_alunos",
        "metrica": "valores_binarios_invalidos",
        "valor": binarios_invalidos,
        "status": "PASS" if binarios_invalidos == 0 else "FAIL"
    },
    {
        "dataset": "avaliacao_alunos",
        "metrica": "inconsistencias_caderno",
        "valor": inconsistencias_caderno,
        "status": "PASS" if inconsistencias_caderno == 0 else "FAIL"
    }
])

In [0]:
# Consolida as métricas críticas de Data Quality utilizadas no monitoramento.

df_monitoramento_qualidade = spark.createDataFrame(
    metricas_qualidade
)

display(
    df_monitoramento_qualidade
    .orderBy("status", "dataset", "metrica")
)

## 5. Visão consolidada de saúde da pipeline

As métricas coletadas nas etapas anteriores são consolidadas para fornecer uma visão operacional do estado atual da pipeline.

Essa visão permite identificar rapidamente falhas de processamento, divergências no Streaming e problemas críticos de qualidade.

In [0]:
# Consolida o status geral dos principais componentes monitorados.

data_hora_monitoramento = datetime.now(timezone.utc)

falhas_batch = (
    df_monitoramento_volume
    .filter(F.col("status") != "SUCESSO")
    .count()
)

falhas_streaming = (
    df_monitoramento_streaming
    .filter(F.col("status") != "SUCESSO")
    .count()
)

falhas_qualidade = (
    df_monitoramento_qualidade
    .filter(F.col("status") == "FAIL")
    .count()
)

resumo_pipeline = [
    {
        "componente": "Batch Silver/Gold",
        "status": "SUCESSO" if falhas_batch == 0 else "FALHA",
        "alertas": falhas_batch,
        "data_hora_utc": data_hora_monitoramento
    },
    {
        "componente": "Streaming",
        "status": "SUCESSO" if falhas_streaming == 0 else "ATENCAO",
        "alertas": falhas_streaming,
        "data_hora_utc": data_hora_monitoramento
    },
    {
        "componente": "Data Quality",
        "status": "SUCESSO" if falhas_qualidade == 0 else "FALHA",
        "alertas": falhas_qualidade,
        "data_hora_utc": data_hora_monitoramento
    }
]

df_saude_pipeline = spark.createDataFrame(resumo_pipeline)

display(df_saude_pipeline)

In [0]:
# Determina o status geral da pipeline a partir dos componentes monitorados.

componentes_com_falha = (
    df_saude_pipeline
    .filter(F.col("status") == "FALHA")
    .count()
)

componentes_com_atencao = (
    df_saude_pipeline
    .filter(F.col("status") == "ATENCAO")
    .count()
)

if componentes_com_falha > 0:
    status_pipeline = "FALHA"

elif componentes_com_atencao > 0:
    status_pipeline = "ATENCAO"

else:
    status_pipeline = "SAUDAVEL"

print(f"Status geral da pipeline: {status_pipeline}")
print(f"Monitoramento realizado em: {data_hora_monitoramento}")

## 6. Regras de alerta

O monitoramento utiliza regras simples para identificar condições que exigem atenção operacional.

Os alertas não alteram automaticamente os dados. Eles sinalizam situações que devem ser investigadas antes da próxima etapa da pipeline.

In [0]:
# Define regras de alerta para volumes, Streaming e Data Quality.

alertas_monitoramento = []

# Streaming divergente
if registros_bronze_stream != registros_silver_stream:
    alertas_monitoramento.append({
        "tipo": "STREAMING",
        "severidade": "ALTA",
        "mensagem": (
            f"Divergência entre Bronze ({registros_bronze_stream}) "
            f"e Silver ({registros_silver_stream})."
        )
    })

# Falhas críticas de qualidade
if falhas_qualidade > 0:
    alertas_monitoramento.append({
        "tipo": "DATA_QUALITY",
        "severidade": "ALTA",
        "mensagem": f"{falhas_qualidade} regra(s) crítica(s) com FAIL."
    })

# Ausência inesperada de dados
objetos_sem_registros = (
    df_monitoramento_volume
    .filter(F.col("registros") == 0)
    .count()
)

if objetos_sem_registros > 0:
    alertas_monitoramento.append({
        "tipo": "VOLUME",
        "severidade": "ALTA",
        "mensagem": f"{objetos_sem_registros} objeto(s) sem registros."
    })

if not alertas_monitoramento:
    print("[OK] Nenhum alerta operacional identificado.")
else:
    display(spark.createDataFrame(alertas_monitoramento))

## 7. Evolução para monitoramento produtivo

Em um ambiente produtivo, as métricas implementadas neste projeto poderiam ser persistidas em uma tabela histórica de monitoramento e integradas a mecanismos de observabilidade e alertas.

Possíveis evoluções incluem:

- armazenamento do histórico de execuções e métricas;
- definição de thresholds de volume por dataset;
- alertas automáticos para falhas e alterações anormais;
- acompanhamento de duração e custo das execuções;
- integração com serviços de observabilidade da AWS;
- dashboards operacionais;
- notificações para regras críticas de Data Quality;
- monitoramento de latência e atraso dos eventos Streaming.

A implementação atual demonstra os controles mínimos necessários para acompanhar a saúde da pipeline sem adicionar complexidade operacional desnecessária ao ambiente acadêmico.